In [117]:
from google import genai
from google.genai import types
import os
import json
from IPython.display import display, Markdown

In [9]:
import qdrant_client
from qdrant_client import models
qdrant_client = qdrant_client.AsyncQdrantClient('http://localhost:6333', timeout=1000)
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-2.5-flash'
# Only run this block for Gemini Developer API
client = genai.Client(api_key=GEMINI_API_KEY)

In [54]:
QUERY = "Qual'è lo stipendio medio di un AI Engineer in Abruzzo con una senority medio alta?"

# Definiamo la fase di query parsing

## Il primo step è la query rewriting

In [55]:
def ask_gemini_to_rewrite_the_query_in_hyve_setup(query: str):
    system_message = """You are an AI language model assistant. Your task is to generate hypothetical passages in less than 300 words in Italian that can answer a given question. If you don't know the answer generate a plausible passage."""
    prompt_template = """Please write a passage to answer the question.
    query: {{query}}
    \n\n"""
    prompt = prompt_template.replace('{{query}}', query)
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3)
    )
    return result.text

    

In [56]:
rewrited_query = ask_gemini_to_rewrite_the_query_in_hyve_setup(QUERY)

In [57]:
rewrited_query

"Determinare lo stipendio medio esatto per un AI Engineer con seniority medio-alta specificamente in Abruzzo può essere complesso, poiché i dati pubblici dettagliati per questa nicchia geografica e professionale sono spesso limitati e soggetti a rapide evoluzioni del mercato. Tuttavia, possiamo fare una stima basandoci sulle tendenze generali del settore in Italia e sulle specificità regionali.\n\nIn generale, un AI Engineer con esperienza consolidata (seniority medio-alta, ovvero tipicamente 5-10+ anni di esperienza) in Italia può aspettarsi una retribuzione annua lorda che varia significativamente. Nelle regioni con un mercato tecnologico più maturo come Lombardia o Lazio, questa fascia può andare dai 50.000€ ai 75.000€, e in alcuni casi superare gli 80.000€ per ruoli molto specifici o di leadership.\n\nPer l'Abruzzo, pur essendo una regione in crescita con un interesse crescente nel settore tech e AI, la media potrebbe essere leggermente inferiore rispetto ai grandi hub, ma comunque

## Il secondo step è l'estrazione delle entità

In [58]:
from pydantic import BaseModel
from typing import List
class Entities(BaseModel):
    entities: List[str]

In [59]:
def ask_gemini_to_extract_entities(query: str):
    system_message = """    You are an helpful and skilled analyst, and you are very good in extracting entities from a given query. 
    You are presented with a text and you are asked to extract entities from it.
    ## GUIDELINES
    1. Extract only Organization names, Locations and person names
    3. Answer in Italian"""
    prompt_template = """Extract entities from the given query: \n\n{{query}}"""
    prompt = prompt_template.replace('{{query}}', query)
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3,
                response_mime_type='application/json',
            response_schema=Entities)
    )
    return result.parsed

In [60]:
entities = ask_gemini_to_extract_entities(QUERY).entities

In [61]:
entities

['Abruzzo']

## Ci possono essere anche altri step come l'estrazione dei timerange o l'estrazione dei filename ma noi non li consideriamo

# Definiamo la fase di retrieval

## Retrieval senza entità e senza Hyve

In [62]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

In [83]:
dense_embeddings = list(dense_embedding_model.embed(QUERY))[0]
bm25_embeddings = list(bm25_embedding_model.embed(QUERY))[0]
late_interaction_embeddings = list(late_interaction_embedding_model.embed(QUERY))[0]

In [84]:
prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [85]:
results_no_entity = [res.payload['document'] for res in  results.points]

In [86]:
results_no_entity[0]

"La valutazione di una RAL di 40.000 euro deve essere necessariamente contestualizzata al costo della vita locale. Mentre a Milano questa cifra permette una vita dignitosa ma limitata dall'incidenza dell'affitto (che per un bilocale può oscillare tra i 1.200 e i 1.400 euro), la medesima retribuzione in città come Roma o Pescara garantisce un potere d'acquisto e una capacità di risparmio notevolmente superiori. 20 In Abruzzo, ad esempio, una RAL di 40.000 euro si traduce in uno stipendio netto mensile di circa 2.124 euro (su 13 mensilità), una cifra che nel contesto locale è considerata di fascia alta, data l'aliquota IRPEF e le addizionali regionali specifiche. 21"

## Retrieval con entità

In [87]:
def create_should_clause(entities: List[str]):
    should = []
    for entity in entities:
        should.append(models.FieldCondition(key='metadata.entities[]', match=models.MatchText(text=entity)))
    return should
    

prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
            filter= models.Filter(should=create_should_clause(entities))
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
            filter= models.Filter(should=create_should_clause(entities))
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [88]:
results_entity = [res.payload['document'] for res in  results.points]

## Retrieval con hyve

In [94]:
dense_embeddings = list(dense_embedding_model.embed(rewrited_query))[0]
bm25_embeddings = list(bm25_embedding_model.embed(rewrited_query))[0]
late_interaction_embeddings = list(late_interaction_embedding_model.embed(rewrited_query))[0]

In [95]:
prefetch = [
        models.Prefetch(
            query=dense_embeddings,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**bm25_embeddings.as_object()),
            using="bm25",
            limit=20,
        ),
    ]

results = await qdrant_client.query_points(
         "pdf",
        prefetch=prefetch,
        query=late_interaction_embeddings,
        using="colbert",
        with_payload=True,
        limit=10,
)

In [96]:
results_hyve = [res.payload['document'] for res in  results.points]

# Definiamo la fase di post retrieval: reranking

In [99]:
results = set(results_entity + results_hyve + results_no_entity)

In [100]:
len(results)

17

In [101]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

C:\Users\andre\PycharmProjects\corso_ai\.venv2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\.cache\huggingface\hub\models--jinaai--jina-reranker-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


modeling.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v3:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [103]:
results = model.rerank(QUERY, list(results))

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

Score: 0.3368
Document: Milano si conferma il polo gravitazionale per l'intelligenza artificiale, ospitando la maggior parte...

Score: 0.2784
Document: Milano si conferma il polo gravitazionale per l'intelligenza artificiale in Italia, ospitando la mag...

Score: 0.1788
Document: - 1. Tech job market and hiring trends in 2025 Michael Page, accesso eseguito il giorno marzo 19, 20...

Score: 0.0903
Document: La valutazione di una Retribuzione Annua Lorda (RAL) di 40.000 euro deve essere necessariamente cont...

Score: 0.0199
Document: La valutazione di una RAL di 40.000 euro deve essere necessariamente contestualizzata al costo della...

Score: -0.0421
Document: Nel mercato competitivo del 2026, lo stipendio base (RAL) è solo una parte del valore totale del pac...

Score: -0.0422
Document: L'esperienza maturata sul campo rimane il driver principale per la definizione della base salariale,...

Score: -0.0451
Document: Il confronto tra le retribuzioni italiane e quelle dei mercati interna

In [112]:
results[6]['document']

'L\'esperienza maturata sul campo rimane il driver principale per la definizione della base salariale, sebbene la rapidità con cui un professionista acquisisce familiarità con i nuovi framework (come LangChain, PyTorch Lightning o Hugging Face) possa accelerare notevolmente gli scatti retributivi. 2  \nQuì c\'era una tabella che è stata sostituita dalla sua descrizione.\nDescrizione della tabella: Questa tabella presenta una panoramica delle retribuzioni e dei bonus stimati in base al livello di seniority. È strutturata in quattro colonne che forniscono informazioni dettagliate per ogni categoria di esperienza.  \nLa prima riga della tabella funge da intestazione e descrive il contenuto di ciascuna colonna. La prima colonna, intitolata "Livello di Seniority", indica il grado di esperienza professionale. La seconda colonna, "RAL Media Nazionale (€)", mostra la Retribuzione Annua Lorda media a livello nazionale espressa in euro. La terza colonna, "Range Minimo-Massimo (€)", specifica l\'

## Definiamo la fase di generazione

In [120]:
def ask_gemini_to_answer_query(query: str, documents: List[str]):
    system_message = """You are an excellent AI assitant that answer user queries given relevant documents. The documents are sorted by relevance (higher means more relevant).
    Rispondi solo con affermazioni fattuali basati sui documenti recuperati.
    Cita le tue fonti alla fine di ogni frase utilizzando [1], [2] ecc
    """
    prompt_template = """Answer the given query using the given context documents: \n\nQUERY: {{query}}. \n\nDOCUMENTS: {{documents}}"""
    prompt = prompt_template.replace('{{query}}', query).replace('{{documents}}', '\n\n'.join(documents))
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                system_instruction=system_message,
                temperature=0.3)
    )
    return result.text

In [130]:
answer = ask_gemini_to_answer_query(QUERY, [result['document'] for result in results[:5]])

In [131]:
display(Markdown(answer))

In Abruzzo, la Retribuzione Annua Lorda (RAL) media per un AI Engineer è compresa tra 20.000 e 35.000 euro [Descrizione della tabella]. Questa fascia di retribuzione rappresenta un differenziale tra -30% e -45% rispetto alla media italiana [Descrizione della tabella]. I documenti non specificano una retribuzione media per un AI Engineer con seniority medio-alta in Abruzzo, ma indicano la RAL media generale per la regione [Descrizione della tabella].

In [129]:
[result['document'] for result in results[:5]]

['Milano si conferma il polo gravitazionale per l\'intelligenza artificiale, ospitando la maggior parte delle multinazionali tech e delle startup ad alta capitalizzazione. Qui, la retribuzione media per un AI Engineer è superiore del 10,5% rispetto alla media nazionale. 4  \nQuì c\'era una tabella che è stata sostituita dalla sua descrizione.\nDescrizione della tabella: La tabella presentata è strutturata in tre colonne distinte.\nLa prima colonna è intitolata "Area Geografica" e indica le diverse località o regioni considerate.\nLa seconda colonna è intitolata "RAL Media AI Engineer (€)" e riporta la Retribuzione Annua Lorda media in euro per un Ingegnere AI.\nLa terza colonna è intitolata "Differenziale vs Media Italia" e mostra la differenza percentuale rispetto alla media italiana della RAL.  \nPassando alla descrizione di ogni riga:  \nLa prima riga della tabella è l\'intestazione e definisce il contenuto delle colonne: "Area Geografica", "RAL Media AI Engineer (€)" e "Differenzia